# Fixed Wing UAV Detection

Author  : Himanshu Raj
 |
Project : Counter-UAS Autonomous Interceptor
 |
Framework : Ultralytics YOLO26

# Import Lib

In [1]:
from ultralytics import YOLO
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import yaml

/home/himanshu-raj/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


# Data Collection & Preprocessing

Dataset Loading

In [2]:
DATA_ROOT = Path("/mnt/5252B43652B420A1/Data centr/IMG_data/Fixed_Wing")

DATA_YAML= DATA_ROOT/"data.yaml"

TRAIN_DIR=  DATA_ROOT/"train"
VALID_DIR= DATA_ROOT/"valid"

Dataset Verification

In [3]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

# Directory Paths
TRAIN_IMAGES = DATA_ROOT / "train" / "images"
TRAIN_LABELS = DATA_ROOT / "train" / "labels"

VALID_IMAGES = DATA_ROOT / "valid" / "images"
VALID_LABELS = DATA_ROOT / "valid" / "labels"


def verify_pairs(image_dir: Path, label_dir: Path):

    images = sorted(
        [img for img in image_dir.iterdir() if img.suffix.lower() in IMAGE_EXTENSIONS]
    )

    missing_labels = []

    for image in images:

        label = label_dir / f"{image.stem}.txt"

        if not label.exists():
            missing_labels.append(image.name)

    print(f"\nDirectory : {image_dir.parent.name}")
    print(f"Images    : {len(images)}")
    print(f"Missing Labels : {len(missing_labels)}")

    if missing_labels:
        print("\nFirst 10 Missing Labels")

        for img in missing_labels[:10]:
            print(img)


with open(DATA_YAML, "r") as file:
    data_config = yaml.safe_load(file)

NUM_CLASSES = data_config["nc"]
CLASS_NAMES = data_config["names"]

print("=" * 60)
print("Dataset Configuration")
print("=" * 60)

print(f"Number of Classes : {NUM_CLASSES}")
print(f"Class Names       : {CLASS_NAMES}")


verify_pairs(TRAIN_IMAGES, TRAIN_LABELS)
verify_pairs(VALID_IMAGES, VALID_LABELS)

Dataset Configuration
Number of Classes : 1
Class Names       : ['FixedWing']

Directory : train
Images    : 1418
Missing Labels : 0

Directory : valid
Images    : 473
Missing Labels : 0


Data Preprocess

# Model Training

Pre-trained Model Selection

In [4]:
MODEL_NAME= "yolo26m.pt"
model=YOLO(MODEL_NAME)

print(f"Lodaded model: {MODEL_NAME}")

Lodaded model: yolo26m.pt


Configuration

In [5]:
TRAIN_CONFIG = {
    # Dataset
    "data": str(DATA_YAML),
    # Model
    "epochs": 60,
    "imgsz": 640,
    # Hardware
    "batch": 2,
    "workers": 2,
    "device": "cpu",
    # Output
    "project": "Fixed_Wing_Detector",
    "name": "fixedwing_v1",
    "exist_ok": True,
    # Optimization
    "optimizer": "AdamW",
    "lr0": 0.0008,
    "lrf": 0.01,
    "weight_decay": 0.0005,
    "momentum": 0.937,
    "cos_lr": True,
    # Early stopping
    "patience": 20,
    # Augmentation
    "mosaic": 1.0,
    "close_mosaic": 10,
    "mixup": 0.10,
    "copy_paste": 0.0,
    "hsv_h": 0.015,
    "hsv_s": 0.40,
    "hsv_v": 0.40,
    "degrees": 5.0,
    "translate": 0.10,
    "scale": 0.50,
    "shear": 0.0,
    "perspective": 0.0,
    "fliplr": 0.50,
    "flipud": 0.0,
    # Validation
    "seed": 42,
    "val": True,
    # Misc
    "cache": False,
    "save": True,
    "plots": True,
    "verbose": True,
    "deterministic": True,
}

Custom Training

In [6]:
result=model.train(**TRAIN_CONFIG)

New https://pypi.org/project/ultralytics/8.4.92 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.74 🚀 Python-3.10.12 torch-2.12.1+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/mnt/5252B43652B420A1/Data centr/IMG_data/Fixed_Wing/data.yaml, degrees=5.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0008, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26m.pt, mome

KeyboardInterrupt: 

# Model Evaluation

In [ ]:
PROJECT_ROOT = Path.cwd().parent

MODEL_DIR = PROJECT_ROOT / "models" / "trained_model"

BEST_MODEL = MODEL_DIR / "fixedwing_yolo26m" / "weights" / "best.pt"

In [ ]:


custom_model = YOLO(str(BEST_MODEL))

print("=" * 60)
print("Best Model Loaded")
print("=" * 60)
print(BEST_MODEL)

In [ ]:
metrics = custom_model.val(
    data=str(DATA_YAML), split="val", imgsz=640, batch=2, device="cpu"
)

Evaluation Metrics

In [ ]:
precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50
map5095 = metrics.box.map

print("=" * 60)
print("Model Evaluation")
print("=" * 60)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"mAP@50    : {map50:.4f}")
print(f"mAP50-95  : {map5095:.4f}")

Graph Visualization

In [ ]:
evaluation_metrics = {
    "Precision": precision,
    "Recall": recall,
    "mAP50": map50,
    "mAP50-95": map5095,
}

plt.figure(figsize=(8, 5))

bars = plt.bar(evaluation_metrics.keys(), evaluation_metrics.values())

plt.ylim(0, 1)

plt.xlabel("Evaluation Metrics")

plt.ylabel("Score")

plt.title("YOLO26m Model Evaluation")

plt.grid(axis="y", linestyle="--", alpha=0.3)

for bar in bars:

    height = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width() / 2, height + 0.02, f"{height:.3f}", ha="center"
    )

plt.show()

# Model Testing

In [ ]:
RESULT_IMAGE = MODEL_DIR / "fixedwing_yolo26m" / "results.png"

image = Image.open(RESULT_IMAGE)

plt.figure(figsize=(15, 10))

plt.imshow(image)

plt.axis("off")

plt.title("Training Results")

plt.show()

Confusion Matrix

In [ ]:
CONFUSION = MODEL_DIR / "fixedwing_yolo26m" / "confusion_matrix.png"

image = Image.open(CONFUSION)

plt.figure(figsize=(8, 8))

plt.imshow(image)

plt.axis("off")

plt.title("Confusion Matrix")

plt.show()

# Model Deployment

In [ ]:
deployment_model = YOLO(str(BEST_MODEL))

print("Deployment Model Loaded Successfully")
print(BEST_MODEL)

In [ ]:
deployment_model.export(format="onnx", dynamic=True, simplify=True)

print("ONNX Model Exported Successfully")